# Trade log — top combos

Full per-combo trade logs for the top-K strategies on the 20%
chronological test partition. Wins are shaded green, losses red.

**Prerequisite:** generate `evaluation/top_strategies.json` first via
`python scripts/runners/run_extract_top_combos.py`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation.composed_strategy_runner import run_strategy, load_test_bars

TOP_STRATEGIES_PATH = REPO / 'evaluation' / 'top_strategies.json'

In [ ]:
payload = json.loads(TOP_STRATEGIES_PATH.read_text())
strategies = payload['top']
bars = load_test_bars()
print(f'Loaded {len(strategies)} strategies, {len(bars):,} test bars')

In [ ]:
results = []
for s in strategies:
    print(f'Running {s["global_combo_id"]}...', flush=True)
    results.append(run_strategy(s, bars=bars))
print('Done.')

In [ ]:
_EXIT_MAP = {'sl': 'SL', 'tp': 'TP', 'time': 'EOD', 'force': 'EOD'}

def render_trade_log(df: pd.DataFrame):
    display_cols = ['date', 'direction', 'entry_px', 'sl_px', 'tp_px', 'contracts',
                    'dollar_risk', 'dollar_reward', 'reason']
    for col in ['exit_px', 'exit_reason', 'actual_pnl']:
        if col in df.columns:
            display_cols.append(col)
    display_cols = [c for c in display_cols if c in df.columns]
    styled = df[display_cols].copy()
    styled['date'] = styled['date'].dt.strftime('%Y-%m-%d')

    if 'exit_reason' in styled.columns:
        styled['exit_reason'] = (styled['exit_reason']
                                 .str.lower()
                                 .map(_EXIT_MAP)
                                 .fillna(styled['exit_reason']))

    fmt = {}
    for col in ['entry_px', 'sl_px', 'tp_px', 'tp_pts', 'gap', 'atr',
                'dollar_risk', 'dollar_reward', 'exit_px']:
        if col in styled.columns:
            fmt[col] = '{:.2f}'
    for col in ['r_open']:
        if col in styled.columns:
            fmt[col] = '{:.4f}'
    for col in ['actual_pnl']:
        if col in styled.columns:
            fmt[col] = '{:.2f}'
    if 'contracts' in styled.columns:
        fmt['contracts'] = '{:.0f}'

    def colour_row(row):
        n = len(row)
        pnl_val = row.get('actual_pnl', None)
        if pd.notna(pnl_val) and str(pnl_val).strip() not in ('', 'nan'):
            try:
                return (['background-color: #c8e6c9; color: #1b5e20'] * n
                        if float(pnl_val) > 0
                        else ['background-color: #ffcdd2; color: #b71c1c'] * n)
            except (ValueError, TypeError):
                pass
        return [''] * n

    try:
        display(styled.style
                .apply(colour_row, axis=1)
                .format(fmt, na_rep='—')
                .hide(axis='index'))
    except Exception:
        print(styled.to_string(index=False))

for r in results:
    display(Markdown(f"### {r['combo_id']} — {len(r['trades'])} trades"))
    if len(r['trades']) > 0:
        render_trade_log(r['trades'])
    else:
        print('(no trades)')